# Constant Station Network (1880–2024)

This notebook builds the fixed/constant station network described in the manuscript:

> To address this bias, we constructed a fixed station network from 1880 onwards, consisting of approximately
> 200–220 stations that remained relatively consistent through time.

**Method:** 
a station counts as active in a given year if it
has a valid (non-missing, quality-controlled) rainfall value on more than 10% of the days that year — but apply
it only to years 1880–2024, where the network has already grown enough that the yearly active count stabilises
at ~200–220 stations. 

sheet `Trend_stations_modify`).

**Input:** the quality-controlled per-station parquet files (`parquet_network_QC`), produced by
`Data_Filtration_clean.ipynb`.

**Output:**
- `active_station_network.xlsx` — one row per year (1880–2024) with the active station count and station list
- `Trend_study/combined_parquet.parquet` — combined daily rainfall for the constant network, ready for the
  Adaptive KDE / WRx-IRx classification notebook

## Step 0 — Config

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Paths (edit these) ---------------------------------------------------
CLEAN_STATION_DIR   = Path("/path/to/parquet_network_QC")          # output of Data_Filtration_clean.ipynb
STATION_LIST_FILE   = "/path/to/5429_Station_list.xlsx"             # Station Number, Lat, Lon
NETWORK_OUTPUT_FILE = "/path/to/active_station_network.xlsx"
TREND_STUDY_DIR     = Path("/path/to/Trend_study")
COMBINED_OUTPUT_FILE = TREND_STUDY_DIR / "combined_parquet.parquet"

TREND_STUDY_DIR.mkdir(parents=True, exist_ok=True)

# ---- Parameters -------------------------------------------------------------
FIXED_NETWORK_START_YEAR = 1880
FIXED_NETWORK_END_YEAR   = 2024
ACTIVE_DAY_COVERAGE_MIN  = 0.10   # a station is "active" in a year if >10% of its days have valid rainfall

## Step 1 — Compute per-year active stations (1880–2024)

Same rule as `Data Filtration.ipynb` cells 27–28: a station is active in a year if more than 10% of that
year's days have a valid (non-missing) rainfall value. Operating on the QC'd data here, so there's no need for
the old notebook's special-casing of recent years / a single quality code.

In [ ]:
station_files = sorted(CLEAN_STATION_DIR.glob("*.parquet"))

yearly_active = {
    year: {"count": 0, "stations": []}
    for year in range(FIXED_NETWORK_START_YEAR, FIXED_NETWORK_END_YEAR + 1)
}

for file_path in station_files:
    try:
        station_number = int(file_path.stem)
    except ValueError:
        continue

    df = pd.read_parquet(file_path)
    df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
    df["Rainfall"] = pd.to_numeric(df["Rainfall"], errors="coerce")

    for year, year_group in df.groupby("Year"):
        year = int(year)
        if year not in yearly_active:
            continue

        total_days = len(year_group)
        valid_days = year_group["Rainfall"].notna().sum()

        if total_days > 0 and (valid_days / total_days) > ACTIVE_DAY_COVERAGE_MIN:
            yearly_active[year]["count"] += 1
            yearly_active[year]["stations"].append(station_number)

network_rows = [
    {
        "Year": year,
        "Station_count": info["count"],
        "Station_Numbers": ",".join(map(str, info["stations"])),
    }
    for year, info in sorted(yearly_active.items())
]
network_df = pd.DataFrame(network_rows)
network_df.to_excel(NETWORK_OUTPUT_FILE, index=False)

print(network_df["Station_count"].describe())
network_df.head()

## Step 2 — Sanity check: does the count stabilise around 200–220 stations?

Matches the manuscript's ~200–220 station claim and `Data Filtration.ipynb` cell 43's plot.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(network_df["Year"], network_df["Station_count"], marker="o", linestyle="-", color="b")
plt.axhspan(200, 220, color="red", alpha=0.15, label="~200-220 station band (manuscript)")
plt.xlabel("Year")
plt.ylabel("Active station count")
plt.title(f"Constant Active Station Network ({FIXED_NETWORK_START_YEAR}-{FIXED_NETWORK_END_YEAR})")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

## Step 3 — Build the combined daily-rainfall file for the constant network

For each year, keep only that year's active stations, merge in Lat/Lon, and build a single combined parquet
file spanning 1880–2024 — this is the file the Adaptive KDE / WRx-IRx notebook reads for the post-1880 period.

In [ ]:
station_coords = pd.read_excel(STATION_LIST_FILE, usecols=["Station Number", "Lat", "Lon"])

combined_frames = []

for _, row in network_df.iterrows():
    year = int(row["Year"])
    if not row["Station_Numbers"]:
        continue
    active_stations = {int(s) for s in row["Station_Numbers"].split(",")}

    for station_number in active_stations:
        file_path = CLEAN_STATION_DIR / f"{station_number}.parquet"
        if not file_path.exists():
            continue

        df = pd.read_parquet(file_path)
        df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
        df = df[df["Year"] == year].copy()
        if df.empty:
            continue

        df["Rainfall"] = pd.to_numeric(df["Rainfall"], errors="coerce")
        df = df[df["Rainfall"].notna() & (df["Rainfall"] != -99.99)]
        if df.empty:
            continue

        df["Station Number"] = station_number
        combined_frames.append(df)

combined_df = pd.concat(combined_frames, ignore_index=True)
combined_df = combined_df.merge(station_coords, on="Station Number", how="left")
combined_df["Date"] = pd.to_datetime(combined_df[["Year", "Month", "Day"]])
combined_df = combined_df.drop(columns=["Year", "Month", "Day"])

combined_df.to_parquet(COMBINED_OUTPUT_FILE, index=False)
print(f"Saved constant-network combined data to {COMBINED_OUTPUT_FILE}")
print(f"Rows: {len(combined_df):,}  |  Stations: {combined_df['Station Number'].nunique():,}")
combined_df.head()